In [6]:
%pip install opencv-python pandas tqdm -q

Note: you may need to restart the kernel to use updated packages.


In [7]:
import os, cv2, numpy as np, pandas as pd
from tqdm import tqdm
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

In [8]:
class SegDataset(Dataset):
    def __init__(self, img_dir, mask_dir=None, transform=None):
        self.img_dir, self.mask_dir = img_dir, mask_dir
        self.fnames = sorted(os.listdir(img_dir))
        self.transform = transform

    def __len__(self):
        return len(self.fnames)

    def __getitem__(self, idx):
        fname = self.fnames[idx]
        img = cv2.imread(os.path.join(self.img_dir, fname))[:, :, ::-1]
        img = cv2.resize(img, (256, 256))
        img = self.transform(img) if self.transform else transforms.ToTensor()(img)

        if self.mask_dir:
            mask = cv2.imread(os.path.join(self.mask_dir, fname), cv2.IMREAD_GRAYSCALE)
            mask = cv2.resize(mask, (256, 256), interpolation=cv2.INTER_NEAREST)
            return img, torch.from_numpy(mask).long()
        return img, fname

In [9]:
def rle_encode(mask):
    pixels = mask.flatten(order="F")
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[:-1:2]
    return " ".join(str(x) for x in runs)

In [10]:
def get_model(num_classes=16):
    model = models.segmentation.deeplabv3_resnet50(pretrained=True)
    model.classifier[4] = nn.Conv2d(256, num_classes, kernel_size=1)
    return model

In [11]:
def train_model(model, train_loader, device, epochs=10):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for imgs, masks in tqdm(train_loader):
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            out = model(imgs)["out"]
            loss = criterion(out, masks)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}, loss={total_loss/len(train_loader):.4f}")
    return model

In [12]:
def inference_and_submit(model, test_loader, device, out_csv="submission.csv"):
    model.eval()
    records = []
    with torch.no_grad():
        for imgs, fnames in tqdm(test_loader):
            imgs = imgs.to(device)
            preds = model(imgs)["out"].argmax(1).cpu().numpy()
            for pred, fname in zip(preds, fnames):
                row = {"img": fname}
                for class_id in range(16):
                    class_mask = (pred == class_id).astype(np.uint8)
                    row[f"class_{class_id}"] = (
                        "none" if class_mask.sum() == 0 else rle_encode(class_mask)
                    )
                records.append(row)
    pd.DataFrame(records).to_csv(out_csv, index=False)
    print(f"Saved {out_csv}")

In [14]:
# 設定路徑
train_img_dir = "./data/train"
train_mask_dir = "./data/train_masks"
test_img_dir = "./data/test"

transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)

# DataLoader
train_set = SegDataset(train_img_dir, train_mask_dir, transform)
train_loader = DataLoader(train_set, batch_size=4, shuffle=True)
test_set = SegDataset(test_img_dir, transform=transform)
test_loader = DataLoader(test_set, batch_size=4, shuffle=False)

# Model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = get_model().to(device)

# 訓練
model = train_model(model, train_loader, device, epochs=5)

# 推論 + 產生 submission.csv
inference_and_submit(model, test_loader, device)

/homes/nfs/ben/project/AI-pruning-project/pruning_project/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/homes/nfs/ben/project/AI-pruning-project/pruning_project/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DeepLabV3_ResNet50_Weights.COCO_WITH_VOC_LABELS_V1`. You can also use `weights=DeepLabV3_ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/deeplabv3_resnet50_coco-cd0a2569.pth" to /homes/nfs/ben/.cache/torch/hub/checkpoints/deeplabv3_resnet50_coco-cd0a2569.pth
100%|██████████| 161M/161M [00:01<00:00, 111MB/s]  
  0%|          | 

TypeError: 'NoneType' object is not subscriptable